<header>
   <p  style='font-size:36px;font-family:Arial; color:#F0F0F0; background-color: #00233c; padding-left: 20pt; padding-top: 20pt;padding-bottom: 10pt; padding-right: 20pt;'>
       Bank ClickStream - Event Transformer: Next-Event Prediction for Customer Journeys
  <br>
       <img id="teradata-logo" src="https://storage.googleapis.com/clearscape_analytics_demo_data/DEMO_Logo/teradata.svg" alt="Teradata" style="width: 125px; height: auto; margin-top: 20pt;">
    </p>
</header>

<p style = 'font-size:20px;font-family:Arial'><b>Introduction</b></p>

<p style = 'font-size:16px;font-family:Arial'>This notebook walks through building a transformer model from scratch to predict the next event in a customer journey sequence (e.g., banking interactions).</p>

<p style = 'font-size:18px;font-family:Arial'><b>Overview</b></p>

<p style = 'font-size:16px;font-family:Arial'><b>Goal</b>: Given a sequence of events like `[login, view_dashboard, check_balance]`, predict what the customer will do next.</p>

<p style = 'font-size:16px;font-family:Arial'><b>Approach</b>:
<li style = 'font-size:16px;font-family:Arial'>1. Build a custom tokenizer using your events as vocabulary</li>
<li style = 'font-size:16px;font-family:Arial'>2. Create a decoder-only transformer (GPT-style) from scratch</li>
<li style = 'font-size:16px;font-family:Arial'>3. Train using causal language modeling (next-token prediction)</li>
<li style = 'font-size:16px;font-family:Arial'>4. Evaluate on a holdout set</li>

<p style = 'font-size:16px;font-family:Arial'><b>Expected Data Format</b>:</p>

<li style = 'font-size:16px;font-family:Arial'>UserId | SessionId | TimeStamp | Event | ContactModality</li>
<p></p>


<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>1. Import packages</b></p>
<p style = 'font-size:16px;font-family:Arial'>Import the teradataml and Python libraries required. </p>

In [ ]:
import os
import json
import math
from typing import List, Dict, Optional, Tuple, Any
from collections import Counter
from dataclasses import dataclass, asdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR
from collections import OrderedDict
from dotenv import load_dotenv, dotenv_values

# Core imports
from teradataml import *
from teradatasqlalchemy.types import *


# Utility imports
import shutil
from IPython.display import clear_output, display as ipydisplay

# Set display options for dataframes, plots, and warnings
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

python_version = "3.11"
print(f'Using Python version {python_version} for user environment')


# a list of required packages to install in the custom OAF container
# modify this if using different models or design patterns
pkgs = ['transformers==4.57.6',
        'torch==2.12.0',
        'pandas==3.0.0',
        'sentence-transformers==5.2.0']

# Optional: Force CPU if MPS causes issues (uncomment if needed)
DEVICE = torch.device('cuda')
# print(f'Forced device: {DEVICE}')

# Optional: Check PyTorch version for MPS compatibility
print(f'PyTorch version: {torch.__version__}')
# if DEVICE.type == 'mps':
#     print('MPS backend is available and will be used for GPU acceleration')

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>2. Connect to TeradataCloud</b></p>
<p style = 'font-size:16px;font-family:Arial'>Connect to TeradataCloud using <code>create_context</code> from the teradataml Python library. If this environment has been prepared for connecting to a TeradataCloud Lake OAF Container, all the details required will be loaded and you will see an acknowledgement after executing this cell.</p>

<p style = 'font-size:18px;font-family:Arial;'><b>2.1 Load the Environment Variables and Connect to Vantage</b></p>
<p style = 'font-size:16px;font-family:Arial;'>Load the environment variables from a .env file and use them to create a connection context to Teradata.</p>

In [ ]:
print("Checking if this environment is ready to connect to TeradataCloud...")

if os.path.exists("/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env"):
    print("Your environment parameter file exist.  Please proceed with this use case.")
    # Load all the variables from the .env file into a dictionary
    env_vars = dotenv_values(
        "/home/jovyan/JupyterLabRoot/TeradataCloud/.config/.env"
    )
    # Create the Context
    eng = create_context(
        host=env_vars.get("host"),
        username=env_vars.get("username"),
        password=env_vars.get("my_variable"),
    )
    execute_sql(
        """SET query_band='DEMO=5._Bank_ClickStream_-_Outcome_Prediction_Model_with_GPT_Transformers;' UPDATE FOR SESSION;"""
    )
    print("Connected to TeradataCloud with:", eng)
else:
    print("Your environment has not been prepared for connecting to TeradataCloud.")
    print("Please contact the support team.")

<hr style='height:1px;border:none;'>

<p style = 'font-size:18px;font-family:Arial;'><b>2.2  Authenticate to the User Environment Service</b></p>

<p style = 'font-size:16px;font-family:Arial;'>To better support integration with Cloud Services and common automation tools; the <b > User Environment Service</b> is accessed via RESTful APIs.  These APIs can be called directly or in the examples shown below that leverage the Python Package for Teradata (teradataml) methods.</p> 

In [ ]:
# We've already loaded all the values into our environment variables and into a dictionary, env_vars.
# username=env_vars.get("username") isn't required when using base_url, pat and pem.

if set_auth_token(
    base_url=env_vars.get("ues_uri"),
    pat_token=env_vars.get("access_token"),
    pem_file=env_vars.get("pem_file"),
    # valid_from=int(time.time()),
):
    print("UES Authentication successful")
else:
    print("UES Authentication failed. Check credentials.")
    sys.exit(1)

<hr style="height:2px;border:none;">

<b style = 'font-size:18px;font-family:Arial;'>3. Create a Custom Container in TeradataCloud</b>

<p style = 'font-size:16px;font-family:Arial;'>If desired, the user can create a <b>new</b> custom environment by starting with a "base" image and customizing it.  The steps are:</p> 
<ul style = 'font-size:16px;font-family:Arial;'>
    <li>List the available "base" images the system supports</li>
    <li>List any existing "custom" environments the user has created</li>
    <li>If there are no custom environments, then create a new one from a base image</li>
    </ul>

In [ ]:
# Check if we have any existing environments
# If any other environments exist along with our default OAF environment, we will delete them
username = env_vars.get("username")
environment_name = username[: min(15, len(username))]

print(
    "Here is a list of the versions of the libraries available to be used within an OAF environments.\n"
)
print(list_base_envs())
env_list = list_user_envs()

if env_list is None:
    print("\nThis user does not have any environments.\nCreating your environment now.")
    demo_env = create_env(
        env_name=f"{environment_name}", base_env="python_3.11", desc="transformers demo env"
    )
    print(demo_env)
else:
    print("\nHere is a list of your current environments:")
    ipydisplay(env_list)
    for env_name in env_list["env_name"]:
        if env_name == environment_name:
            demo_env = get_env(environment_name)
            print(
                "Your default environment already exists. You can continue with this notebook.\n\n"
            )
        else:
            print(
                f"Your existing environment, {env_name} doesn't match our default environment for this user."
            )
            print("We're going to delete it.")
            print(f"Please wait: Environment {env_name} is being removed!")
            remove_env(env_name)

<p style = 'font-size:16px;font-family:Arial'>Check version for local packages</p>

In [ ]:
!pip list | grep transformers
!pip list | grep torch
!pip list | grep numpy
!pip list | grep pandas

<p style = 'font-size:16px;font-family:Arial'>Install the required packages in the virtual env and check the status of the installation</p>

In [ ]:
lib_claim_id = pd.DataFrame()
lib_claim_id = demo_env.install_lib(["transformers==5.5.0", 
                                     "torch==2.13.0",
                                     "pandas==2.2.3",
                                     "numpy==1.23.5"])
print("Libraries Installed") 

In [ ]:
#Get the status of the libraries installation
demo_env.status(str(lib_claim_id["Claim Id"].iloc[0]))

<hr style="height:2px;border:none">
<p style = 'font-size:20px;font-family:Arial'><b>3. Load Your Data</b></p>

<p style = 'font-size:16px;font-family:Arial'>Load your event pathing data. Expected columns:</p>
<li style = 'font-size:16px;font-family:Arial'>`UserId`: Customer identifier</li>
<li style = 'font-size:16px;font-family:Arial'>`SessionId`: Session/journey identifier</li>
<li style = 'font-size:16px;font-family:Arial'>`TimeStamp`: When the event occurred</li>
<li style = 'font-size:16px;font-family:Arial'>`Event`: The event name (this becomes our vocabulary)</li>
<li style = 'font-size:16px;font-family:Arial'>`ContactModality`: Channel (web, mobile, etc.) - optional</li>

In [ ]:
df = df = DataFrame(in_schema("DEMO_Bank","Session_Events"))
df

<hr style="height:2px;border:none">
<p style="font-size:20px;font-family:Arial"><b>4. Create a python script and execute it using the Apply Class</b></p>

<p style="font-size:16px;font-family:Arial">
Set the session to the GPU Analytic compute group you desire otherwise it will set to default. The cluster needs to be running to execute the APPLY class.</p>

In [ ]:
gpu_compute_group = env_vars.get("gpu_compute_group")
execute_sql(f"SET SESSION COMPUTE GROUP {gpu_compute_group};")
print(f"Compute group set to {gpu_compute_group}") 

<p style="font-size:16px;font-family:Arial">
Install the python script file to the environment using install_file(). The python file contains the functionality of creating tranformer model and predict next events in the customer journey.
</p>

<p style = 'font-size:16px;font-family:Arial'>The file <code>"clickstream_transformers.py"</code> follows the below steps.</p>

<li style = 'font-size:16px;font-family:Arial'><b>1. Build the Tokenizer</b>: The tokenizer maps event names to integer IDs. Unlike NLP where we use subword tokenization, here each event is a single token.</li><p>

<li style = 'font-size:16px;font-family:Arial'><b>2. Prepare PyTorch Datasets</b>: For next-token prediction, we create input-target pairs:
    <p>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Input: [BOS, e1, e2, ..., en-1]<br>
    &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Target: [e1, e2, ..., en, EOS]</p>
At each position, the model learns to predict the next token.</li><p>

<li style = 'font-size:16px;font-family:Arial'><b>3. Build the Transformer Model</b>:We build a decoder-only transformer (GPT-style) with:<p>
<p><b>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Token Embeddings</b>: Convert token IDs to dense vectors<br>
    <b>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Positional Encodings</b>: Add position information (learned or sinusoidal)<br>
    <b>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Transformer Blocks</b>: Multi-head self-attention + feed-forward networks<br>
    <b>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Output Projection</b>: Project back to vocabulary size for next-token prediction</p></li> <p></p>

<li style = 'font-size:16px;font-family:Arial'><b>4. Training Loop</b>:We train the model using:<p>
<p><b>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Loss</b>: Cross-entropy (next-token prediction)<br>
    <b>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Optimizer</b>: AdamW with weight decay<br>
    <b>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Learning Rate</b>: Warmup + cosine annealing<br>
    <b>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Early Stopping</b>: Stop if validation loss doesn't improve</li> <p>
    
<li style = 'font-size:16px;font-family:Arial'><b>5. Evaluate on Test Set</b>:Evaluate the trained model on the held-out test set using:<p>
        <p><b>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Loss & Perplexity</b>: How well does it model the sequences?<br>
            <b>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;Accuracy@K</b>: Is the true next event in the top K predictions?<br>
            <b>&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;MRR(Mean Reciprocal Rank)</b>: Average of 1/rank of correct prediction</li><p>    

In [ ]:
demo_env.install_file(file_path ="clickstream_transformers.py", replace=True)


<p style="font-size:16px;font-family:Arial">
The APPLY class executes your Python script directly within the user environment, enabling in-database processing at scale. It reads text from each row of the dataset, performs Named Entity Recognition (NER) using Hugging Face language models accelerated by NVIDIA GPUs, and outputs the extracted entities into a structured dataframe in just seconds. </p>  

<p style="font-size:16px;font-family:Arial"> In this example, we use <code>tner/roberta-large-ontonotes5</code>, a general-purpose NER model trained on the OntoNotes 5 dataset. It supports entities like ORG, PERSON, DATE, PRODUCT, and GPE. However, it's important to note that the “PRODUCT” entity in OntoNotes refers primarily to physical products (e.g., iPhone, Windows OS), not financial instruments (e.g., Roth IRA, mutual funds). To improve financial domain accuracy, this model can be further fine-tuned on domain-specific data to better recognize investment products, insurance types, and retirement accounts.
 </p> 

In [ ]:
df_new = df.select(['UserID','SessionID','Event_TS', 'ContactModality', 'Event']).head(100)
df_new

In [ ]:
apply_obj = Apply(data = df_new,
                  apply_command = 'python clickstream_transformers.py',
                  # returns = {"UserID": VARCHAR(64000), "SessionID": VARCHAR(64000), "Event_TS": VARCHAR(64000), "ContactModality": VARCHAR(64000), "Event": VARCHAR(64000)},
                  returns = {"seq": VARCHAR(64000)
                            ,"event": VARCHAR(64000)
                             , "prob": VARCHAR(64000)
                            },
                  env_name = f'{environment_name}',
                  delimiter = ",",
                  # quotechar = '|'
                 )

import time

start = time.time()

# Execute the Python script inside the remote user environment.
df_out = apply_obj.execute_script()

print(f'Time: {time.time() - start}')

df_out.head(20)


<hr style='height:2px;border:none'>
<b style = 'font-size:20px;font-family:Arial'>5. Summary</b>

<p style = 'font-size:16px;font-family:Arial'>In this notebook, we:</p>

<li style = 'font-size:16px;font-family:Arial'><b>Loaded event pathing data</b> with UserId, SessionId, TimeStamp, Event columns</li>
<li style = 'font-size:16px;font-family:Arial'><b>Built a custom tokenizer</b> that converts events to token IDs
<li style = 'font-size:16px;font-family:Arial'><b>Created PyTorch datasets</b> for next-token prediction
<li style = 'font-size:16px;font-family:Arial'><b>Implemented a transformer from scratch</b>: with<br>
   &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- Token embeddings + positional encodings<br>
   &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- Multi-head self-attention with causal masking<br>
   &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- Feed-forward networks<br>
   &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;- Layer normalization and residual connections<br></li>
       <li style = 'font-size:16px;font-family:Arial'><b>Trained the model</b> with warmup, cosine LR schedule, and early stopping</li>
    <li style = 'font-size:16px;font-family:Arial'><b>Evaluated</b> using loss, perplexity, accuracy@K, and MRR</li>
    <li style = 'font-size:16px;font-family:Arial'><b>Made predictions</b> for next events in sequences</li>
    <li style = 'font-size:16px;font-family:Arial'><b>Saved</b> the model for later use</li><p>

<p style = 'font-size:18px;font-family:Arial'><b>Next Steps</b></p>

<li style = 'font-size:16px;font-family:Arial'><b>Tune hyperparameters</b>: Try different model sizes, learning rates, dropout</li>
    <li style = 'font-size:16px;font-family:Arial'><b>Add features</b>: Include ContactModality, time between events, user features</li>
    <li style = 'font-size:16px;font-family:Arial'><b>Evaluate more</b>: Look at predictions by event type, sequence length</li>
    <li style = 'font-size:16px;font-family:Arial'><b>Deploy</b>: Create an API endpoint for real-time predictions</li><p>

<hr style='height:2px;border:none'>
<b style = 'font-size:20px;font-family:Arial'>6. Cleanup</b>
<p style = 'font-size:18px;font-family:Arial'><b>Work Tables</b></p>
<p style = 'font-size:16px;font-family:Arial'>Cleanup the OAF User Environment storage. If you will be executing other TeradataCloud OAF notebooks, you can skip this step.</p>

In [ ]:
#Remove the existing user environment 
from IPython.display import display, HTML
try:
    result = remove_env(environment_name)
    print("Environment removed!")
except Exception as e:
    print("Could not remove the environment!")
    print("Error:", str(e))

<p style = 'font-size:16px;font-family:Arial'>Please delete your database connection.</p>

In [ ]:
try:
    result = remove_context()
    print("Context removed!")
except Exception as e:
    print("Could not remove the Context!")
    print("Error:", str(e))

<footer style="padding-bottom:35px; border-bottom:3px solid">
  <div style="float:right; margin-top:14px">Copyright © Teradata - 2026. All Rights Reserved</div>
</footer>